# Belief-persistence evaluation on Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Zip the extracted images and upload the zip to Google Drive at `MyDrive/cmpe249/`.
   From the repo root, with the positive and control images extracted under `data/nuscenes/samples/CAM_FRONT/` (see the README):
   ```bash
   python -m zipfile -c data/nuscenes_samples.zip data/nuscenes/samples
   ```
   Upload the resulting `data/nuscenes_samples.zip` (1,254 images, about 185 MB).
   Its paths must start with `samples/CAM_FRONT/`, which zipping `samples` from inside `data/nuscenes` gives.

Results are written to that Drive folder, and `run` skips finished (sequence, mode) pairs, so if Colab disconnects or the free-tier GPU quota runs out, reconnect later and run the cells again; it continues where it stopped.
Every result line records the run settings and package versions, and `run` refuses to resume a file written with different ones.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/cmpe249"   # holds nuscenes_samples.zip; results go here too
BRANCH = "main"

# Run once per model: Qwen first, then switch to the LLaVA line and run from step 2 again.
# MAX_SIDE = longest image side fed to the model. LLaVA-OneVision tiles each image, so a 6-frame
# history at 1024 px is ~27.7k tokens and runs out of memory on a T4; 768 px is ~14.9k.
# (Qwen at 1024 px is ~5.3k.) Keep MAX_SIDE fixed for a model once its run has started.
# MODEL, TAG, MAX_SIDE = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b", 1024
MODEL, TAG, MAX_SIDE = "llava-hf/llava-onevision-qwen2-7b-ov-hf", "llava7b", 768

RESULTS = f"{DRIVE_DIR}/results_{TAG}_v2.jsonl"   # v2: rerun with the "most recent image" prompt
RUN = (f"PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m belief_eval run --dataroot data/nuscenes "
       f"--model {MODEL} --4bit --max-side {MAX_SIDE}")

def progress():
    """Finished records per (sequence kind, mode) in the results file."""
    import collections, json, os
    if os.path.exists(RESULTS):
        counts = collections.Counter((r.get("kind", "positive"), r["mode"]) for r in map(json.loads, open(RESULTS)))
        for (kind, mode), n in sorted(counts.items()):
            print(f"{kind:9} {mode:12} {n}")

## 2. Setup: Drive, code, packages, images

Package versions are pinned in `requirements.txt` (torch stays at Colab's preinstalled build).

In [3]:
from google.colab import drive, userdata
drive.mount("/content/drive")

# The Colab secret is only readable from this kernel, not from the `!python ...` processes below;
# exported as an env var, they inherit it. Optional: without it, downloads are just rate-limited.
import os
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:  # secret missing or notebook access not granted
    print(f"HF_TOKEN not set ({type(e).__name__}); downloading anonymously")

!if [ -d /content/repo ]; then git -C /content/repo fetch -q && git -C /content/repo checkout -q {BRANCH} && git -C /content/repo pull -q; else git clone -q -b {BRANCH} https://github.com/n-zad/av-vlm-belief-persistence.git /content/repo; fi
%cd /content/repo
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -E "^(torch|transformers|accelerate|bitsandbytes) "
!mkdir -p data/nuscenes && unzip -q -o {DRIVE_DIR}/nuscenes_samples.zip -d data/nuscenes

# every image the sequences use must be at data/nuscenes/samples/CAM_FRONT/
import json, os
for name in ("sequences_trainval.jsonl", "sequences_control.jsonl"):
    imgs = {fr["image"] for s in map(json.loads, open(name)) for fr in s["frames"]}
    found = sum(os.path.exists(os.path.join("data/nuscenes", i)) for i in imgs)
    print(f"{name}: {found} / {len(imgs)} images present" + ("" if found == len(imgs) else "  <-- missing images"))

Mounted at /content/drive
/content/repo
accelerate                            1.16.0
bitsandbytes                          0.50.2
torch                                 2.14.1+cu130
transformers                          5.24.1
sequences_trainval.jsonl: 804 / 804 images present
sequences_control.jsonl: 450 / 450 images present


## 3. Smoke test: one sequence, all five modes

Check that the raw answers are sensible JSON before the long run.
If they are empty or gibberish, stop here and report it.
If you get a CUDA out-of-memory error, lower `MAX_SIDE` in step 1 (e.g. 640).
Planted frames (`planted=True`) are fixed answers written into the history, not model output.

In [4]:
!rm -f /content/smoke.jsonl
!{RUN} --sequences sequences_trainval.jsonl --limit 1 --out /content/smoke.jsonl

import json
for rec in map(json.loads, open("/content/smoke.jsonl")):
    print(f"--- {rec['mode']} (evidence frame {rec['evidence_index']})")
    for f in rec["frames"]:
        print(f"t={f['t']} present={f['present']} action={f['action']} parsed_from={f['parsed_from']} "
              f"planted={f['planted']} | {f['raw'][:120]!r}")
print(rec["config"])

config.json: 100% 2.53k/2.53k [00:00<00:00, 1.53MB/s]
model.safetensors.index.json: 100% 78.0k/78.0k [00:00<00:00, 137MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 60.7M/16.1G [00:04<16:52, 15.8MB/s, 46.6MB/s  ]
Reconstructing (incomplete total...):   3% 549M/16.1G [00:11<05:10, 50.0MB/s,  140MB/s  ]
Reconstructing (incomplete total...):  16% 2.65G/16.1G [01:03<05:24, 41.3MB/s, 40.6MB/s  ]
Reconstructing (incomplete total...):  18% 2.93G/16.1G [01:15<03:06, 70.2MB/s, 70.2MB/s  ]
Reconstructing (incomplete total...):  22% 3.55G/16.1G [01:42<09:43, 21.4MB/s, 21.7MB/s  ]
Reconstru

## 4. Full run, in priority order

Each step is safe to re-run after a disconnect.
Run them in order: if the GPU quota runs out, what is left unfinished is the least important part.
If you change `MAX_SIDE` after a partial run, delete that model's results file in Drive first (the run refuses to mix settings).

**4a. Core results** (stateless, stateful, planted "no"): 145 sequences × 3 modes.
Each positive mode should reach 145.

In [5]:
!{RUN} --sequences sequences_trainval.jsonl --modes stateless stateful planted_no --out {RESULTS}
progress()

Loading weights: 100% 765/765 [01:03<00:00, 12.06it/s] 
100% 435/435 [00:00<?, ?seq/s]
results -> /content/drive/MyDrive/cmpe249/results_llava7b_v2.jsonl
positive  planted_no   145
positive  planted_yes  41
positive  reconsider   41
positive  stateful     145
positive  stateless    145


**4b. Planted "yes" and the reconsider mitigation.**

In [6]:
!{RUN} --sequences sequences_trainval.jsonl --modes planted_yes reconsider --out {RESULTS}
progress()

Loading weights: 100% 765/765 [01:03<00:00, 12.07it/s] 
100% 290/290 [4:10:58<00:00, 72.40s/seq, reconsider scene-1100_7a88efef_14]
results -> /content/drive/MyDrive/cmpe249/results_llava7b_v2.jsonl
positive  planted_no   145
positive  planted_yes  145
positive  reconsider   145
positive  stateful     145
positive  stateless    145


**4c. Control sequences without history** (pedestrian-free frames: the false-positive rate).

In [7]:
!{RUN} --sequences sequences_control.jsonl --modes stateless --out {RESULTS}
progress()

Loading weights: 100% 765/765 [01:03<00:00, 12.02it/s] 
100% 75/75 [26:44<00:00, 21.39s/seq, stateless scene-1074_control_21]
results -> /content/drive/MyDrive/cmpe249/results_llava7b_v2.jsonl
control   stateless    75
positive  planted_no   145
positive  planted_yes  145
positive  reconsider   145
positive  stateful     145
positive  stateless    145


**4d. Planted "yes" on control sequences** (does a wrong "yes" persist on empty frames?).

In [8]:
!{RUN} --sequences sequences_control.jsonl --modes planted_yes --out {RESULTS}
progress()

Loading weights: 100% 765/765 [01:03<00:00, 12.05it/s] 
100% 75/75 [1:20:04<00:00, 64.05s/seq, planted_yes scene-1074_control_21]
results -> /content/drive/MyDrive/cmpe249/results_llava7b_v2.jsonl
control   planted_yes  75
control   stateless    75
positive  planted_no   145
positive  planted_yes  145
positive  reconsider   145
positive  stateful     145
positive  stateless    145


## 5. Scores (all models run so far)

Includes median latency, longest prompt and peak GPU memory per mode.

In [9]:
!python -m belief_eval score {DRIVE_DIR}/results_*_v2.jsonl

{
  "Qwen/Qwen2.5-VL-7B-Instruct": {
    "planted_no": {
      "n_sequences": 145,
      "n_wrong_before": 145,
      "persistence_rate": 0.9586206896551724,
      "persistence_rate_ci": [
        0.922077922077922,
        0.9863013698630136
      ],
      "latency_median": 0.5,
      "latency_p90": 4,
      "n_never_corrected": 127,
      "post_evidence_accuracy": 0.07436781609195402,
      "post_evidence_accuracy_ci": [
        0.038356164383561646,
        0.11519607843137254
      ],
      "frame_accuracy": 0.07436781609195402,
      "says_present_rate": 0.08170515097690942,
      "safety_balanced_accuracy": 0.49725274725274726,
      "unsafe_action_rate": 0.06216696269982238,
      "switches": {
        "yes_to_no": 7,
        "no_to_yes": 21
      },
      "accuracy_by_offset": {
        "0": {
          "accuracy": 0.06206896551724138,
          "n": 145
        },
        "1": {
          "accuracy": 0.041379310344827586,
          "n": 145
        },
        "2": {
          